In [ ]:
import numpy as np

from rsvd_correction.matrix_generators import (
    DiagonalKnownSpectrum,
    ExactLowRank,
    ExponentialDecay,
    PolynomialDecay,
    SignalPlusNoise,
)

from benchmark import run_benchmark, print_results
from hypothesis_test import run_hypothesis_test

In [2]:
N     = 5000
K     = 555
P     = 20
SIGMA = np.array([10.0 / (i + 1) for i in range(K)])
SEED  = 42

# polynomial_decay
ALPHA = 1.0

# exponential_decay
BETA = 0.5

# signal_plus_noise
NOISE_LEVEL  = 1.0

# hypothesis test
N_TRIALS   = 100
HT_CONFIGS = [
    # --- correction helps (high noise, low-to-moderate c) ---
    (600,  2, 2.0),
    (600,  3, 2.0),
    (600,  5, 2.0),
    (1000, 2, 5.0),
    (1000, 3, 5.0),
    (1000, 5, 5.0),
    # --- borderline (moderate noise, moderate c) ---
    (600,  3, 1.0),
    (1000, 5, 1.0),
    (1000, 8, 2.0),
    # --- correction hurts (low noise or high c) ---
    (600,  2, 0.5),
    (600,  5, 0.5),
    (600,  8, 2.0),
    (600, 12, 5.0),
    (1000, 12, 5.0),
]

In [ ]:
generators = [
    ExactLowRank(sigma=SIGMA),
    DiagonalKnownSpectrum(sigma=SIGMA),
    PolynomialDecay(alpha=ALPHA),
    ExponentialDecay(beta=BETA),
    SignalPlusNoise(sigma_signal=SIGMA, noise_level=NOISE_LEVEL),
]

for gen in generators:
    A, sigma_true = gen(n=N, k=K, seed=SEED)
    print_results(run_benchmark(gen.name, A, sigma_true, k=K, p=P, seed=SEED))

In [8]:
run_hypothesis_test(HT_CONFIGS, n_trials=N_TRIALS, p=P)

     N     K     c  noise |      D_bar        s_D        t        p |        CI (-inf, hi) | reject?
----------------------------------------------------------------------------------------------------
   600   280  2.00    2.0 |    -0.2163     0.0323  -67.013   0.0000 | (-inf,  -0.2110) |     YES
   600   180  3.00    2.0 |    -0.2097     0.0547  -38.364   0.0000 | (-inf,  -0.2006) |     YES
   600   100  5.00    2.0 |    -0.3024     0.1313  -23.023   0.0000 | (-inf,  -0.2806) |     YES
  1000   480  2.00    5.0 |    -1.6141     0.1249 -129.285   0.0000 | (-inf,  -1.5934) |     YES
  1000   313  3.00    5.0 |    -1.6133     0.1618  -99.681   0.0000 | (-inf,  -1.5864) |     YES
  1000   180  5.00    5.0 |    -1.9831     0.4604  -43.075   0.0000 | (-inf,  -1.9066) |     YES
   600   180  3.00    1.0 |     0.2443     0.0471   51.915   1.0000 | (-inf,   0.2521) |      no
  1000   180  5.00    1.0 |     0.1914     0.0409   46.812   1.0000 | (-inf,   0.1982) |      no
  1000   105  8.00    